# Raman Spectroscopy — Consensus-Based Quality Control Classification

**Goal:** Use the five consensus spectra as a reference standard to flag raw milk samples that fall outside the expected spectral envelope, indicating potential quality or adulteration concerns.

**Input data:**
- `raman_consensus_spectra.xlsx` — five consensus curves built from 42 quality-controlled processed milk spectra
- `raman_preprocessed.xlsx` — the 42 preprocessed training spectra (used to calibrate the L2 threshold)
- `src of raw milk/` — 6 raw milk spectra (Samples 1–6, 532 nm)

**Pipeline:**
1. Load & preprocess raw milk spectra with the identical pipeline used on training data
2. Calibrate per-consensus threshold from the L2 distance distribution of training spectra
3. Compute L2 distances of each raw milk sample to each consensus
4. Classify each sample (pass / marginal / fail) per consensus and by majority vote
5. Pointwise deviation analysis — identify which spectral regions drive any exceedances
6. Per-region QC breakdown aligned to known milk Raman markers

> **Data availability:** The raw LabRAM `.txt` spectra are not included in this repository because the final datasets are provided in `data/`. This notebook reads the raw milk spectra from `src of raw milk/`, plus the unaligned `raman_consensus_spectra.xlsx` and `raman_preprocessed.xlsx`; the consensus spectra are provided (aligned) as `data/raman_consensus_spectra_aligned_402_3499.xlsx`. This notebook is included to document the processing steps; its file paths refer to the original project layout.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from scipy.signal import savgol_filter
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

CONSENSUS_COLORS = {
    'Arithmetic Mean':        'crimson',
    'B-spline Mean':          'darkorange',
    'Robust Median Centre':   'green',
    'Geometric Median':       'darkgreen',
    'Wasserstein Barycenter': 'purple',
}
SAMPLE_COLORS = cm.tab10(np.linspace(0, 0.6, 6))

# Known milk Raman markers (same reference table as decomposition notebook)
MARKERS = [
    (605.2,  'Lipid',   'C=O deformation (triglycerides)'),
    (874.8,  'Lipid',   'C–C stretch –(CH₂)ₙ–'),
    (1001.5, 'Protein', 'Phe ring breathing (casein/whey)'),
    (1062.6, 'Lipid',   'Skeletal C–C stretch (all-trans FA)'),
    (1079.0, 'Lipid',   'C–C stretch FA / lactose C–O'),
    (1123.1, 'Lipid',   'Glycerol ester C–C/C–O–C'),
    (1155.7, 'Lactose', 'C–O stretch (lactose marker)'),
    (1265.5, 'Lipid',   'δ(=C–H) unsaturated FA / Amide III'),
    (1300.7, 'Lipid',   'CH₂ twisting (sat. FA)'),
    (1438.5, 'Lipid',   'CH₂ scissoring (dominant lipid)'),
    (1521.4, 'Lipid',   'Carotenoid C=C (β-carotene)'),
    (1652.5, 'Protein', 'Amide I / C=C unsaturated FA'),
    (1743.8, 'Lipid',   'C=O ester carbonyl (TG/FFA)'),
    (2851.7, 'Lipid',   'νs CH₂ (fatty acid chains)'),
    (2891.2, 'Lipid',   'νas CH₂ (fatty acid chains)'),
    (2929.1, 'Lipid',   'νas CH₃ / Fermi resonance'),
    (3171.8, 'Water',   'O–H stretch (H-bonded water)'),
]
CAT_COLOR = {'Lipid': '#e08030', 'Protein': '#3070b0', 'Lactose': '#40a060', 'Water': '#9060c0'}

## 1. Preprocessing Pipeline

Identical to `raman_consensus.ipynb`: Savitzky-Golay smoothing → ALS baseline correction → non-negative clip → L² normalisation.

In [ ]:
def sg_smooth(y, window=5, polyorder=2):
    return savgol_filter(y, window_length=window, polyorder=polyorder)

def als_baseline(y, lam=1e5, p=0.01, n_iter=10):
    L = len(y)
    D = diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L), format='csc')
    H = lam * D.T.dot(D)
    w = np.ones(L)
    for _ in range(n_iter):
        W = diags(w, 0, format='csc')
        Z = spsolve(W + H, w * y)
        w = np.where(y > Z, p, 1 - p)
    return Z

def l2_normalize(y):
    n = np.linalg.norm(y)
    return y / n if n > 0 else y

def preprocess(y):
    y = sg_smooth(y)
    y = y - als_baseline(y)
    y = np.clip(y, 0, None)
    return l2_normalize(y)

## 2. Load Raw Milk Spectra & Apply Preprocessing

In [ ]:
RAW_DIR = 'src of raw milk'

raw_files = sorted(glob.glob(os.path.join(RAW_DIR, '*.txt')))
print(f'Found {len(raw_files)} raw milk spectra:')
for f in raw_files:
    print(f'  {os.path.basename(f)}')

raw_spectra = {}
wn_raw = None

for fpath in raw_files:
    sample_name = os.path.basename(fpath).split(' 532nm')[0]   # e.g. 'Sample 1'
    exposure    = '8s' if '8s' in fpath else '10s'
    label       = f'{sample_name} ({exposure})'

    with open(fpath, 'rb') as fh:
        text = fh.read().decode('latin-1')
    lines = [l for l in text.splitlines() if not l.startswith('#') and l.strip()]
    data  = np.array([[float(v) for v in l.split('\t')] for l in lines])

    if wn_raw is None:
        wn_raw = data[:, 0]
    raw_spectra[label] = data[:, 1]

# Apply preprocessing
prep_spectra = {label: preprocess(y) for label, y in raw_spectra.items()}
sample_names = list(prep_spectra.keys())

print(f'\nPreprocessing complete.')
print(f'Wavenumber range : {wn_raw.min():.1f} – {wn_raw.max():.1f} cm⁻¹')
print(f'Data points      : {len(wn_raw)}')
print(f'\nNote: Samples 5–6 were acquired at 8 s exposure vs 10 s for Samples 1–4.'
      '\nL² normalisation makes absolute intensity differences immaterial, but peak'
      '\nshape differences from different SNR may still be present.')

## 3. Load Consensus Spectra & Training L2 Distances

In [ ]:
# Consensus spectra
cons_df  = pd.read_excel('raman_consensus_spectra.xlsx', index_col=0)
x_cons   = cons_df.index.values.astype(float)
cons_methods = list(cons_df.columns)

# Training preprocessed spectra (42 quality-controlled samples)
train_df = pd.read_excel('raman_preprocessed.xlsx', index_col=0)
F_train  = train_df.values   # (2118, 42)

# Interpolate raw spectra onto the consensus grid
# (grids differ by up to ~0.15 cm⁻¹ due to instrument calibration offset)
from scipy.interpolate import interp1d
def regrid(y_raw, wn_from, wn_to):
    return interp1d(wn_from, y_raw, kind='linear', bounds_error=False,
                   fill_value='extrapolate')(wn_to)

# Re-grid already-preprocessed spectra onto consensus axis
prep_spectra = {label: regrid(y, wn_raw, x_cons)
                for label, y in prep_spectra.items()}
x = x_cons  # all subsequent operations use consensus axis
print(f'Regridded raw spectra onto consensus axis ({len(x)} points)')

# Per-consensus L2 distance distribution from training set
# used to calibrate the QC threshold
train_l2 = {}   # dict: method -> array of 42 L2 distances
for method in cons_methods:
    c = cons_df[method].values
    dists = np.linalg.norm(F_train - c[:, None], axis=0)
    train_l2[method] = dists

print('Training L2 distance statistics per consensus method:')
print(f'{"Method":<28} {"Mean":>8} {"Std":>8} {"P95":>8} {"Max":>8}')
for method, dists in train_l2.items():
    print(f'{method:<28} {dists.mean():8.4f} {dists.std():8.4f} '
          f'{np.percentile(dists,95):8.4f} {dists.max():8.4f}')

## 4. QC Threshold Definition

Three threshold levels are defined from the training L2 distribution:

| Level | Formula | Interpretation |
|-------|---------|----------------|
| **Pass** | L2 ≤ mean + 1 × std | Within normal variation of QC training set |
| **Marginal** | mean + 1σ < L2 ≤ mean + 2σ | Elevated but not conclusive |
| **Fail** | L2 > mean + 2 × std | Outside expected spectral envelope — flag for review |

In [ ]:
thresholds = {}
for method, dists in train_l2.items():
    mu, sigma = dists.mean(), dists.std()
    thresholds[method] = {
        'pass_limit':     mu + 1.0 * sigma,
        'marginal_limit': mu + 2.0 * sigma,
        'mean':           mu,
        'std':            sigma,
    }

print('QC thresholds per consensus method:')
print(f'{"Method":<28} {"Mean L2":>9} {"Pass (μ+1σ)":>12} {"Marginal (μ+2σ)":>16}')
for method, t in thresholds.items():
    print(f'{method:<28} {t["mean"]:9.4f} {t["pass_limit"]:12.4f} {t["marginal_limit"]:16.4f}')

## 5. Visual Overview — Raw Milk vs Consensus

In [ ]:
fig, axes = plt.subplots(len(cons_methods), 1, figsize=(14, 4 * len(cons_methods)), sharex=True)

for ax, method in zip(axes, cons_methods):
    c    = cons_df[method].values
    t    = thresholds[method]
    std_train = train_l2[method].std()

    # Pointwise envelope: consensus ± mean_L2 × (1σ / ||c||), as a rough band
    # More principled: use pointwise std of training spectra around consensus
    pw_std = np.std(F_train - c[:, None], axis=1)  # pointwise residual std

    ax.fill_between(x, c - 2 * pw_std, c + 2 * pw_std,
                    color=CONSENSUS_COLORS[method], alpha=0.10, label='±2σ band (training)')
    ax.fill_between(x, c - pw_std, c + pw_std,
                    color=CONSENSUS_COLORS[method], alpha=0.18, label='±1σ band (training)')
    ax.plot(x, c, color=CONSENSUS_COLORS[method], linewidth=1.6, label=method)

    for i, (label, y) in enumerate(prep_spectra.items()):
        ax.plot(x, y, linewidth=0.8, color=SAMPLE_COLORS[i], alpha=0.75,
                linestyle='--', label=label)

    ax.set_ylabel('Norm. Intensity')
    ax.set_title(method)
    ax.legend(fontsize=7, ncol=4, loc='upper right')

axes[-1].set_xlabel('Raman Shift (cm⁻¹)')
plt.suptitle('Raw Milk Spectra vs Consensus Spectra (±1σ / ±2σ pointwise bands)', fontsize=13, y=1.005)
plt.tight_layout()
plt.show()

## 6. L2 Distance Computation & Classification

In [ ]:
def classify(dist, t):
    if dist <= t['pass_limit']:
        return 'Pass'
    elif dist <= t['marginal_limit']:
        return 'Marginal'
    else:
        return 'Fail'

CLASS_COLOR = {'Pass': '#2ecc71', 'Marginal': '#f39c12', 'Fail': '#e74c3c'}

# Build distance & classification table
records = []
for label, y in prep_spectra.items():
    row = {'Sample': label}
    for method in cons_methods:
        c    = cons_df[method].values
        dist = np.linalg.norm(y - c)
        cls  = classify(dist, thresholds[method])
        row[f'{method}_L2']    = round(dist, 5)
        row[f'{method}_Class'] = cls
    records.append(row)

results_df = pd.DataFrame(records).set_index('Sample')

# Majority vote across the five consensus methods
def majority_vote(row):
    classes = [row[f'{m}_Class'] for m in cons_methods]
    n_fail     = classes.count('Fail')
    n_marginal = classes.count('Marginal')
    if n_fail >= 2:
        return 'Fail'
    elif n_fail == 1 or n_marginal >= 2:
        return 'Marginal'
    else:
        return 'Pass'

results_df['Majority Vote'] = results_df.apply(majority_vote, axis=1)

# Display
l2_cols  = [f'{m}_L2'    for m in cons_methods]
cls_cols = [f'{m}_Class' for m in cons_methods]
print('=== L2 Distances ===')
print(results_df[l2_cols].to_string())
print('\n=== Classifications ===')
print(results_df[cls_cols + ['Majority Vote']].to_string())

## 7. Classification Summary Heatmap

In [ ]:
CLASS_NUM = {'Pass': 0, 'Marginal': 1, 'Fail': 2}
cols_to_plot = cons_methods + ['Majority Vote']
col_labels   = [m.replace(' ', '\n') for m in cons_methods] + ['Majority\nVote']

heat_data = np.array([
    [CLASS_NUM[results_df.loc[s, f'{m}_Class']] for m in cons_methods] +
    [CLASS_NUM[results_df.loc[s, 'Majority Vote']]]
    for s in sample_names
])

from matplotlib.colors import ListedColormap
cmap_qc = ListedColormap(['#2ecc71', '#f39c12', '#e74c3c'])

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(heat_data, cmap=cmap_qc, vmin=-0.5, vmax=2.5, aspect='auto')

ax.set_xticks(range(len(cols_to_plot)))
ax.set_xticklabels(col_labels, fontsize=9)
ax.set_yticks(range(len(sample_names)))
ax.set_yticklabels(sample_names, fontsize=9)

# Annotate cells
for i, s in enumerate(sample_names):
    for j, m in enumerate(cons_methods):
        cls  = results_df.loc[s, f'{m}_Class']
        dist = results_df.loc[s, f'{m}_L2']
        ax.text(j, i, f'{cls}\n{dist:.3f}', ha='center', va='center', fontsize=7)
    mv = results_df.loc[s, 'Majority Vote']
    ax.text(len(cons_methods), i, mv, ha='center', va='center', fontsize=8, fontweight='bold')

# Separate majority vote column with a vertical line
ax.axvline(len(cons_methods) - 0.5, color='black', linewidth=2)

# Colorbar
cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2])
cbar.ax.set_yticklabels(['Pass', 'Marginal', 'Fail'])

ax.set_title('QC Classification — Raw Milk Samples vs Consensus Spectra\n'
             '(cell text: class label / L2 distance)', fontsize=11)
plt.tight_layout()
plt.show()

## 8. L2 Distance Distribution — Raw Milk vs Training Set

In [ ]:
fig, axes = plt.subplots(1, len(cons_methods), figsize=(16, 4), sharey=False)

for ax, method in zip(axes, cons_methods):
    t     = thresholds[method]
    train = train_l2[method]

    # Training distribution
    ax.hist(train, bins=15, color=CONSENSUS_COLORS[method], alpha=0.4, label='Training (42 spectra)')

    # Threshold lines
    ax.axvline(t['pass_limit'],     color='#f39c12', linewidth=1.5, linestyle='--', label='μ+1σ (pass limit)')
    ax.axvline(t['marginal_limit'], color='#e74c3c', linewidth=1.5, linestyle='--', label='μ+2σ (fail limit)')

    # Raw milk samples
    for i, (label, y) in enumerate(prep_spectra.items()):
        c    = cons_df[method].values
        dist = np.linalg.norm(y - c)
        cls  = classify(dist, t)
        ax.axvline(dist, color=CLASS_COLOR[cls], linewidth=1.2,
                   linestyle='-', alpha=0.9,
                   label=f'{label} [{cls}]')
        ax.annotate(label.split()[1], xy=(dist, 0.5), xycoords=('data', 'axes fraction'),
                    fontsize=7, rotation=90, color=CLASS_COLOR[cls], va='bottom', ha='right')

    ax.set_xlabel('L2 distance')
    ax.set_ylabel('Count')
    ax.set_title(method.replace(' ', '\n'), fontsize=8)
    ax.legend(fontsize=6, loc='upper right')

plt.suptitle('L2 Distance Distributions: Training Set vs Raw Milk Samples', fontsize=12)
plt.tight_layout()
plt.show()

## 9. Pointwise Deviation Analysis

For each raw milk sample the signed deviation from each consensus is computed pointwise. Regions where the deviation persistently exceeds the ±2σ pointwise training band are the spectral drivers of any QC exceedance.

In [ ]:
# Precompute pointwise residual std of training spectra around each consensus
pw_std_dict = {}
for method in cons_methods:
    c = cons_df[method].values
    pw_std_dict[method] = np.std(F_train - c[:, None], axis=1)

# Plot per sample — deviation from Grand Mean consensus (most interpretable)
ref_method = 'Arithmetic Mean'
c_ref    = cons_df[ref_method].values
pw_std   = pw_std_dict[ref_method]

fig, axes = plt.subplots(len(sample_names), 1, figsize=(14, 3.5 * len(sample_names)), sharex=True)

for ax, (label, y) in zip(axes, prep_spectra.items()):
    dev = y - c_ref
    cls = results_df.loc[label, f'{ref_method}_Class']

    ax.fill_between(x, -2 * pw_std, 2 * pw_std,
                    color='grey', alpha=0.15, label='±2σ training band')
    ax.fill_between(x, -pw_std, pw_std,
                    color='grey', alpha=0.25, label='±1σ training band')
    ax.plot(x, dev, color=SAMPLE_COLORS[sample_names.index(label)],
            linewidth=0.9, label=f'{label}  [{cls}]')
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')

    # Mark known marker positions
    for wn, cat, _ in MARKERS:
        ax.axvline(wn, color=CAT_COLOR[cat], linewidth=0.5, linestyle=':', alpha=0.5)

    ax.set_ylabel('Δ Intensity')
    ax.set_title(f'{label}  [vs {ref_method}:  {cls}]', fontsize=9)
    ax.legend(fontsize=7, loc='upper right')

axes[-1].set_xlabel('Raman Shift (cm⁻¹)')
plt.suptitle(f'Pointwise Deviation from {ref_method} — Raw Milk Samples\n'
             '(dotted lines = known milk markers; bands = training ±1σ / ±2σ)',
             fontsize=12, y=1.005)
plt.tight_layout()
plt.show()

## 10. Per-Region QC Analysis

The spectrum is divided into chemically meaningful regions. The fractional contribution of each region to the total L2 exceedance is computed for each sample, identifying which chemical domains (lipid, protein, lactose, water) are driving any anomaly.

In [ ]:
REGIONS = {
    'Fingerprint\n(400–900)':    (400,  900),
    'Protein/Lactose\n(900–1200)': (900,  1200),
    'Lipid fingerprint\n(1200–1500)': (1200, 1500),
    'Amide / C=C\n(1500–1800)':  (1500, 1800),
    'Silent region\n(1800–2700)': (1800, 2700),
    'C–H stretch\n(2700–3050)':  (2700, 3050),
    'O–H stretch\n(3050–3500)':  (3050, 3500),
}

ref_method = 'Arithmetic Mean'
c_ref    = cons_df[ref_method].values
pw_std   = pw_std_dict[ref_method]

region_dev = pd.DataFrame(index=sample_names, columns=list(REGIONS.keys()), dtype=float)

for label, y in prep_spectra.items():
    dev = (y - c_ref) ** 2   # squared deviation
    for reg_name, (lo, hi) in REGIONS.items():
        mask = (x >= lo) & (x <= hi)
        region_dev.loc[label, reg_name] = np.sum(dev[mask])

# Normalise each sample row to fractional contribution
region_frac = region_dev.div(region_dev.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Stacked bar — fractional contribution per region
ax = axes[0]
bottom = np.zeros(len(sample_names))
reg_colors = cm.Set2(np.linspace(0, 1, len(REGIONS)))
for (reg, _), color in zip(REGIONS.items(), reg_colors):
    vals = region_frac[reg].values.astype(float)
    ax.bar(range(len(sample_names)), vals, bottom=bottom,
           label=reg.replace('\n', ' '), color=color, alpha=0.85)
    bottom += vals
ax.set_xticks(range(len(sample_names)))
ax.set_xticklabels([s.replace(' (', '\n(') for s in sample_names], fontsize=8)
ax.set_ylabel('Fractional squared deviation')
ax.set_title(f'Region-wise Deviation Fraction\n(from {ref_method})')
ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left')

# Heatmap — absolute squared deviation per region
ax = axes[1]
im = ax.imshow(region_dev.values.astype(float), aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(REGIONS)))
ax.set_xticklabels([r for r in REGIONS], fontsize=7)
ax.set_yticks(range(len(sample_names)))
ax.set_yticklabels(sample_names, fontsize=8)
ax.set_title(f'Absolute Squared Deviation per Region\n(from {ref_method})')
plt.colorbar(im, ax=ax, label='Σ(y−c)²')
for i in range(len(sample_names)):
    for j in range(len(REGIONS)):
        ax.text(j, i, f'{region_dev.values[i, j]:.4f}',
                ha='center', va='center', fontsize=6)

plt.suptitle('Per-Region Spectral Deviation Analysis', fontsize=12)
plt.tight_layout()
plt.show()

## 11. Cross-Consensus Stability of QC Verdict

For samples flagged as Marginal or Fail, this section shows whether the verdict is consistent across all five consensus methods or driven by only one.

In [ ]:
# Radar / spider chart: L2 distance of each sample normalised by the fail threshold
# Values > 1 mean the sample has exceeded the fail threshold for that consensus

import matplotlib.patches as mpatches

N = len(cons_methods)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, axes = plt.subplots(2, 3, figsize=(14, 9),
                         subplot_kw=dict(polar=True))
axes = axes.ravel()

for ax, (label, y) in zip(axes, prep_spectra.items()):
    cls_mv = results_df.loc[label, 'Majority Vote']
    values = []
    for method in cons_methods:
        c     = cons_df[method].values
        dist  = np.linalg.norm(y - c)
        t_val = thresholds[method]['marginal_limit']  # normalise by fail threshold
        values.append(dist / t_val)
    values += values[:1]

    ax.plot(angles, values, linewidth=1.8,
            color=CLASS_COLOR[cls_mv], label=cls_mv)
    ax.fill(angles, values, color=CLASS_COLOR[cls_mv], alpha=0.25)

    # Reference circles at 0.5 (well within pass), 1.0 (fail boundary)
    for r, style, lbl in [(0.5, ':', 'half-threshold'), (1.0, '--', 'fail threshold')]:
        ax.plot(angles, [r] * (N + 1), linestyle=style, color='grey',
                linewidth=0.8, label=lbl)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([m.replace(' ', '\n') for m in cons_methods], fontsize=7)
    ax.set_title(f'{label}\n[{cls_mv}]', fontsize=9, pad=12)
    ax.set_ylim(0, max(1.5, max(values) * 1.1))

# Hide the 6th subplot if even
if len(prep_spectra) < len(axes):
    for ax in axes[len(prep_spectra):]:
        ax.set_visible(False)

handles = [
    mpatches.Patch(color=CLASS_COLOR['Pass'],     label='Pass'),
    mpatches.Patch(color=CLASS_COLOR['Marginal'], label='Marginal'),
    mpatches.Patch(color=CLASS_COLOR['Fail'],     label='Fail'),
]
fig.legend(handles=handles, loc='lower right', fontsize=9)
plt.suptitle('Cross-Consensus QC Radar Charts\n'
             '(radial axis = L2 / fail-threshold; dashed circle = fail boundary)',
             fontsize=12, y=1.005)
plt.tight_layout()
plt.show()

## 12. DWT D5:D6 Energy Ratio Method

The Discrete Wavelet Transform (db4, 6 levels) decomposes each spectrum into detail bands at different scales:

| Level | Approx. feature width | Physical meaning |
|-------|-----------------------|-----------------|
| **D5** | ~46 cm⁻¹ | Moderate-width peaks — lipid C–C stretches, ester C=O, carotenoid bands |
| **D6** | ~93 cm⁻¹ | Broad features — CH₂/CH₃ stretch cluster envelope, Amide I region |

The ratio **E(D5) / E(D6)** measures the relative energy in moderate-scale vs broad-scale spectral structure. A quality-controlled milk spectrum has a characteristic ratio established from the 42-sample training set.

**Threshold:** two-sided — a sample is flagged if its ratio falls outside mean ± 1σ (marginal) or mean ± 2σ (fail) of the training distribution. Both extremes are anomalous: an unusually high ratio indicates excess narrow-peak structure (possible adulteration introducing new sharp bands); an unusually low ratio indicates broad-feature dominance (possible fluorescence residual or unusual compositional shift).

In [ ]:
import pywt

DWT_WAVELET = 'db4'
DWT_LEVELS  = 6

def dwt_d5_d6_ratio(y, wavelet=DWT_WAVELET, levels=DWT_LEVELS):
    """Return E(D5) / E(D6) for a 1-D signal y.

    With pywt.wavedec at `levels` levels the coefficient list is:
        [cA_n, cD_n, cD_{n-1}, ..., cD_1]
    so index 1 = cD6 (coarsest detail) and index 2 = cD5.
    """
    coeffs = pywt.wavedec(y, wavelet, level=levels)
    e_d6 = np.sum(coeffs[1] ** 2)   # cD6
    e_d5 = np.sum(coeffs[2] ** 2)   # cD5
    return e_d5 / e_d6 if e_d6 > 0 else np.nan

# ── Calibrate threshold from training spectra ─────────────────────────────────
train_ratios = np.array([dwt_d5_d6_ratio(F_train[:, j]) for j in range(F_train.shape[1])])

dwt_mean  = train_ratios.mean()
dwt_std   = train_ratios.std()
dwt_pass  = (dwt_mean - dwt_std,   dwt_mean + dwt_std)    # ±1σ pass band
dwt_marg  = (dwt_mean - 2*dwt_std, dwt_mean + 2*dwt_std)  # ±2σ marginal band

print(f'Training D5:D6 ratio  —  mean: {dwt_mean:.5f}  std: {dwt_std:.5f}')
print(f'  Pass band     (±1σ): [{dwt_pass[0]:.5f}, {dwt_pass[1]:.5f}]')
print(f'  Marginal band (±2σ): [{dwt_marg[0]:.5f}, {dwt_marg[1]:.5f}]')
print()

# ── Classify raw milk samples ─────────────────────────────────────────────────
def classify_dwt(ratio):
    if dwt_marg[0] <= ratio <= dwt_marg[1]:
        if dwt_pass[0] <= ratio <= dwt_pass[1]:
            return 'Pass'
        return 'Marginal'
    return 'Fail'

dwt_results = {}
for label, y in prep_spectra.items():
    r   = dwt_d5_d6_ratio(y)
    cls = classify_dwt(r)
    dwt_results[label] = {'ratio': r, 'class': cls}
    print(f'{label:<22}  ratio = {r:.5f}  →  {cls}')

# Store onto results_df for the combined summary
results_df['DWT D5:D6 Ratio']       = [dwt_results[s]['ratio'] for s in sample_names]
results_df['DWT D5:D6 Class']       = [dwt_results[s]['class'] for s in sample_names]

## 13. Combined Summary Report

In [ ]:
all_methods_cols = [f'{m}_Class' for m in cons_methods] + ['DWT D5:D6 Class']
all_labels       = cons_methods + ['DWT D5:D6']

# Extended majority vote including DWT
def majority_vote_extended(row):
    classes    = [row[f'{m}_Class'] for m in cons_methods] + [row['DWT D5:D6 Class']]
    n_fail     = classes.count('Fail')
    n_marginal = classes.count('Marginal')
    if n_fail >= 2:
        return 'Fail'
    elif n_fail == 1 or n_marginal >= 2:
        return 'Marginal'
    else:
        return 'Pass'

results_df['Extended Vote'] = results_df.apply(majority_vote_extended, axis=1)

print('=' * 80)
print('COMBINED QC CLASSIFICATION REPORT — RAW MILK SAMPLES')
print('=' * 80)
print(f'  Methods : 5 × consensus L2 distance  +  DWT D5:D6 energy ratio')
print(f'  Pass    : within μ±1σ of training distribution')
print(f'  Marginal: between μ±1σ and μ±2σ')
print(f'  Fail    : outside μ±2σ')
print()
print(f'{"Sample":<22} {"L2 Vote":>9} {"DWT":>9} {"Ext. Vote":>10}')
print('-' * 80)
for s in sample_names:
    l2_mv  = results_df.loc[s, 'Majority Vote']
    dwt_c  = results_df.loc[s, 'DWT D5:D6 Class']
    ext_mv = results_df.loc[s, 'Extended Vote']
    agree  = '✓' if l2_mv == ext_mv else '△'
    print(f'{s:<22} {l2_mv:>9} {dwt_c:>9} {ext_mv:>10}  {agree}')

print()
print('Key:  ✓ = DWT agrees with L2 majority   △ = DWT changes verdict')
print()

for verdict in ['Fail', 'Marginal', 'Pass']:
    samps = [s for s in sample_names if results_df.loc[s, 'Extended Vote'] == verdict]
    print(f'  {verdict:<9} ({len(samps)}): {samps}')

print()
print(f'DWT D5:D6 ratio — training: mean={dwt_mean:.4f}, std={dwt_std:.4f}')
print(f'  Pass band  (±1σ): [{dwt_pass[0]:.4f}, {dwt_pass[1]:.4f}]')
print(f'  Fail limit (±2σ): [{dwt_marg[0]:.4f}, {dwt_marg[1]:.4f}]')
print()
for s in sample_names:
    r = results_df.loc[s, 'DWT D5:D6 Ratio']
    c = results_df.loc[s, 'DWT D5:D6 Class']
    delta = r - dwt_mean
    print(f'  {s:<22}  ratio={r:.4f}  Δ={delta:+.4f}  [{c}]')
print('=' * 80)

## 14. Save Results

In [ ]:
out_path = 'raman_qc_results.xlsx'
with pd.ExcelWriter(out_path) as writer:
    results_df.to_excel(writer, sheet_name='QC Classification')
    region_dev.to_excel(writer, sheet_name='Region Deviation')
    region_frac.to_excel(writer, sheet_name='Region Deviation Fraction')

    # L2 threshold table
    thresh_rows = [
        {'Method': m, 'Mean L2': t['mean'], 'Std L2': t['std'],
         'Pass limit (μ+1σ)': t['pass_limit'], 'Fail limit (μ+2σ)': t['marginal_limit']}
        for m, t in thresholds.items()
    ]
    pd.DataFrame(thresh_rows).to_excel(writer, sheet_name='L2 Thresholds', index=False)

    # DWT threshold table
    dwt_thresh = pd.DataFrame([{
        'Method':            'DWT D5:D6 ratio',
        'Wavelet':           DWT_WAVELET,
        'Levels':            DWT_LEVELS,
        'Training mean':     dwt_mean,
        'Training std':      dwt_std,
        'Pass lower (μ-1σ)': dwt_pass[0],
        'Pass upper (μ+1σ)': dwt_pass[1],
        'Fail lower (μ-2σ)': dwt_marg[0],
        'Fail upper (μ+2σ)': dwt_marg[1],
    }])
    dwt_thresh.to_excel(writer, sheet_name='DWT Thresholds', index=False)

print(f'Saved → {out_path}')